# 03 · Qwen3.5-0.8B Task-only 기준선
01의 동일 episode split·정규화·query 간격 8을 사용합니다. Teacher cache 없이 demonstration L1으로 학습합니다.
최근 2 query × 2 camera, 8차원 proprio, 연속 action `[1,8,7]` 계약입니다. Teacher와 별도 Python 3.13 환경을 만듭니다.
실행 순서: 환경 → Qwen 가중치 다운로드 → 1-step → 작은 episode overfit → 결과 확인 후 본 학습/재개.
실제 0.8B 가중치·GPU 학습은 아직 검증하지 않았습니다. 이 노트북을 실행하면서 검증 결과를 기록합니다.
[고정 Qwen revision](https://huggingface.co/Qwen/Qwen3.5-0.8B/tree/2fc06364715b967f1860aea9cf38778875588b17).
Task-only/KD의 공정 비교에는 같은 초기 revision·seed·head·sampler·최종 step·입력 budget을 유지하십시오.
이 노트북에는 KD loss, CoTinyVLA Plan/Think 생성, 환경 rollout 성공률 평가가 포함되어 있지 않습니다.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = "db79038305323f4eb5328e5986c87f889fd0e74158ee31fc9de8f944bb418d9e"
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'))
def run(args, env=None):
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('실행에 실패했습니다. 위 오류를 확인하십시오. 후속 학습은 시작하지 않습니다.')
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

In [ ]:
STUDENT_PY = environment('vlade_qwen_task_py313', '3.13', [
    'torch==2.12.0', 'torchvision==0.27.0', 'transformers==5.9.0',
    'accelerate', 'pillow', 'numpy', 'sentencepiece'])
run([STUDENT_PY, '-c', "import torch; from transformers import Qwen3_5Model; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,torch.cuda.get_device_name(0))"])
(ROOT / 'student_environment_freeze.txt').write_text(subprocess.check_output(
    [str(UV), 'pip', 'freeze', '--python', str(STUDENT_PY)], text=True))

## Qwen 초기 가중치 준비
이 셀을 실행하면 공식 고정 revision의 모델 가중치 약 1.75 GB와 processor 자료를 Drive에 받습니다. 기존 로컬 폴더의 processor 자료만으로는 전체 학습을 실행할 수 없습니다. 오늘 다운로드한 것으로 기록하지 않습니다.

In [ ]:
QWEN = ROOT / 'checkpoints/students/qwen3_5_0_8b'
program = "import sys; from pathlib import Path; sys.path.insert(0,'src'); from vlade.data.colab_assets import download; download('Qwen/Qwen3.5-0.8B','2fc06364715b967f1860aea9cf38778875588b17',Path(sys.argv[1]))"
run([STUDENT_PY, '-c', program, QWEN])

In [ ]:
PREPARED = ROOT / 'data/processed/libero_plus_episode80'
from vlade.data.episode_data import EpisodeData, write_json
from vlade.training.run_control import epoch_budget, overfit_report, require_reports, planned_checkpoint_space, pilot_context
MAIN_EPOCHS = 1
RUN_MAIN_STUDENT = False
OVERFIT_MAX_RATIO = .8  # 초기/마지막 loss 중앙값의 비율: 파일럿 감소 검사 기준
SAVES_PER_EPOCH = 4
BUDGET = epoch_budget(EpisodeData(PREPARED), MAIN_EPOCHS, batch_size=1, accumulation=8)
RUN_NAME = 'qwen-task-epoch-seed42'
CONFIGS = ROOT / 'run-configs'
CONFIGS.mkdir(exist_ok=True)
student = json.loads((CODE / 'configs/models/student/qwen3_5_0_8b.json').read_text())['student']
base = dict(method='task_only', data=str(PREPARED), initial_checkpoint=str(QWEN), student=student,
            seed=42, batch_size=1, accumulation_steps=8, max_steps=BUDGET['optimizer_steps'],
            save_steps=max(1,(BUDGET['optimizer_steps']+MAIN_EPOCHS*SAVES_PER_EPOCH-1)//(MAIN_EPOCHS*SAVES_PER_EPOCH)),
            learning_rate=2e-5, weight_decay=.01, validation_queries=32, eval_steps=100, episode_limit=0,
            budget=BUDGET,
            output=str(ROOT / 'runs/student_task' / RUN_NAME))
def student_config(suffix, **updates):
    settings = {**base, **updates}
    settings['output'] = str(ROOT / 'runs/student_task' / (RUN_NAME + suffix))
    path = CONFIGS / (RUN_NAME + suffix + '.json')
    if path.exists() and json.loads(path.read_text()) != settings:
        raise ValueError('기존 실행 설정 불일치: RUN_NAME을 바꾸십시오.')
    path.write_text(json.dumps(settings, indent=2))
    return path, Path(settings['output'])
def train_student(config_path, output):
    settings = json.loads(config_path.read_text())
    run([STUDENT_PY, 'scripts/train/run_student_task.py', '--config', config_path, '--preflight'])
    checkpoints = completed_checkpoints(output)
    latest = checkpoints[-1] if checkpoints else None
    if latest and int(latest.name.split('-')[1]) >= settings['max_steps']:
        print('이미 완료된 학습:', latest)
        return
    command = [STUDENT_PY, 'scripts/train/run_student_task.py', '--config', config_path]
    if latest:
        command += ['--resume', latest]
    run(command)

In [ ]:
smoke_config, smoke_output = student_config('-one-step', max_steps=1, save_steps=1,
    accumulation_steps=1, validation_queries=1)
train_student(smoke_config, smoke_output)
run([STUDENT_PY, 'scripts/train/check_student_checkpoint.py', '--checkpoint', smoke_output / 'step-0000001',
     '--data', PREPARED, '--config', smoke_config, '--report', smoke_output / 'reload_report.json'])

## 작은 episode overfit 검사
같은 train episode를 반복하며 처음/마지막 5개 loss 중앙값의 감소 비율을 자동 검사합니다. 이 파일럿 기준은 완전 수렴이나 rollout 성공률을 의미하지 않습니다. 실패 보고서가 있으면 본 학습을 거부합니다. 실제 가중치의 입력 모달리티 영향은 별도 확인 대상입니다.

In [ ]:
overfit_config, overfit_output = student_config('-overfit', max_steps=50, save_steps=50,
    accumulation_steps=1, episode_limit=1, validation_queries=4)
train_student(overfit_config, overfit_output)
by_step = {}
for log in sorted(overfit_output.glob('metrics-*.jsonl'), key=lambda path: path.stat().st_mtime):
    for line in log.read_text().splitlines():
        row = json.loads(line)
        by_step[row['step']] = row
report = overfit_report(list(by_step.values()), max_ratio=OVERFIT_MAX_RATIO)
overfit_steps = json.loads(overfit_config.read_text())['max_steps']
report['context'] = pilot_context(overfit_config, overfit_output / f'step-{overfit_steps:07d}',
    PREPARED, 'student', policy={'max_ratio': OVERFIT_MAX_RATIO, 'window': 5})
write_json(overfit_output / 'overfit_report.json', report)
print('파일럿 loss 감소 검사:', report)
import matplotlib.pyplot as plt
for log in sorted(overfit_output.glob('metrics-*.jsonl')):
    rows = [json.loads(line) for line in log.read_text().splitlines()]
    plt.plot([row['step'] for row in rows], [row['loss'] for row in rows], label=log.stem)
plt.xlabel('Optimizer step'); plt.ylabel('Train L1'); plt.legend(); plt.show()

In [ ]:
MAIN_CONFIG, MAIN_OUTPUT = student_config('')
if RUN_MAIN_STUDENT:
    overfit_steps = json.loads(overfit_config.read_text())['max_steps']
    require_reports([smoke_output / 'reload_report.json', overfit_output / 'overfit_report.json'],
        [pilot_context(smoke_config, smoke_output / 'step-0000001', PREPARED, 'student'),
         pilot_context(overfit_config, overfit_output / f'step-{overfit_steps:07d}', PREPARED, 'student',
            policy={'max_ratio': OVERFIT_MAX_RATIO, 'window': 5})])
    checkpoints = completed_checkpoints(MAIN_OUTPUT)
    print('저장 공간 계획:', planned_checkpoint_space(ROOT, smoke_output / 'step-0000001',
        base['max_steps'], base['save_steps'], int(checkpoints[-1].name.split('-')[1]) if checkpoints else 0))
    train_student(MAIN_CONFIG, MAIN_OUTPUT)
else:
    print('본 학습 대기: budget·reload·overfit 결과 확인 후 RUN_MAIN_STUDENT=True로 실행하십시오.')
reports = [json.loads(path.read_text()) for path in sorted(MAIN_OUTPUT.glob('run-*.json'), key=lambda path: path.stat().st_mtime_ns)]
print(json.dumps(reports[-1] if reports else {}, indent=2, ensure_ascii=False))
print('Validation L1은 rollout 성공률이 아닙니다. 환경 평가와 최종 기준 성능 기록은 별도로 진행하십시오.')